# Reading a hidden bit from three qubits: a GReLU neuron on the source-sign states

**The data.** A chain of 50 qubits starts in its ground state. Qubit 25 is then given a quarter turn about the z axis, either one way (s = +1) or the other (s = −1). The chain evolves in time under the XXX (Heisenberg) Hamiltonian. The direction s is a hidden classical bit. The question is whether a small block of qubits near qubit 25 still shows which way it was turned.

**The task in this notebook (the simplest version).** XXX chain only. The block is the three qubits 24, 25, 26 (k = 3, centred on the turned qubit: d = 0), at times t = 0.1 to 7.5. Each sample is the 8 × 8 density matrix ρ of those three qubits, labelled s = ±1. One quantized GReLU neuron is trained with **Algorithm 8** (squared loss, exact gradient) to read s from ρ.

**Steps.**
1. **The neuron**: GReLU, H(θ) built from Pauli terms, the exact loss and gradient, and firing (Algorithm 5).
2. **The data**: cut the three-qubit states out of the full-chain files, convert them to the right index order, and check that order against values the simulation stored itself.
3. **Training**: Adam, with θ recorded at every step.
4. **Testing**: exact outputs, then classification by firing the neuron with finite copies.
5. **Figures.**

Paper: He, Liu & Wilde, *Fermi–Dirac machines as quantizations of neurons* (arXiv:2605.24386). Dataset docs: `data/source_sign_mps_N50/README.md` and `DATASET.md`.

**Running.** Use the "Julia 1.12" kernel, from anywhere inside the repository. The whole notebook takes about a minute; the first run also cuts the region states with the dataset's Python tool (~30 s, needs numpy and h5py). Julia packages: HDF5, Optimisers, Plots, SpecialFunctions. CSVs go to `results/` and figures to `figures/`.

**The same code as scripts.** `neuron.jl`, `data.jl`, `run.jl` and `plots.jl` hold exactly the code cells below (`julia 4_source_sign_grelu_minimal/run.jl` runs everything from the repository root).

In [ ]:
# Find this folder from wherever the notebook was started: walk up until we reach the repository root
# (the folder that contains data/source_sign_mps_N50).
function find_root(dir=pwd())
  while !isdir(joinpath(dir, "data", "source_sign_mps_N50"))
    parent = dirname(dir)
    parent == dir && error("start the notebook from inside the repository")
    dir = parent
  end
  return dir
end
const HERE = joinpath(find_root(), "4_source_sign_grelu_minimal")
println("this folder: ", HERE)

## 1. The neuron

### Activation
GReLU_T(x) = x Φ(x/T) + T φ(x/T) is a ReLU whose corner is smoothed by a Gaussian of width T.

In [ ]:
# neuron.jl — the quantized GReLU neuron for a small region of k qubits (here k = 3, so every matrix is 8 × 8).
#
# What the neuron does, in plain words:
#   * It has one weight θ_j for each Pauli term H_j: the identity I (a bias), Z on each qubit, and ZZ, XX, YY on
#     each pair of neighbouring qubits.
#   * The weights build a Hamiltonian  H(θ) = Σ_j θ_j H_j.
#   * Its output on a region state ρ is  Tr[GReLU_T(H(θ)) ρ]:  the activation applied to H(θ), then averaged
#     over the state.
#   * It predicts s = +1 when the output is above 1/2, and s = −1 otherwise.
#
# Training (Algorithm 8 of He, Liu & Wilde, arXiv:2605.24386):
#   loss L(θ) = average over samples of (output − target)²,   target = 1 for s = +1 and 0 for s = −1.
# Adam lowers this loss using its exact gradient, which is what the Algorithm 8 circuit would give after
# infinitely many runs.

using LinearAlgebra, Random, Statistics
using SpecialFunctions: erf

# ---- the activation function ----
Phi(x) = (1 + erf(x / sqrt(2))) / 2            # standard normal cumulative distribution
phi(x) = exp(-x^2 / 2) / sqrt(2pi)             # standard normal bell curve
grelu(x, T) = x * Phi(x / T) + T * phi(x / T)  # smoothed ReLU

### Pauli strings and H(θ)
Every Pauli string is built with `kron`, which puts the **first** qubit of the region in the **most significant** bit of the matrix index. The region states are converted to this same order when they are loaded (Section 2), so "qubit 1" always means chain site 24, the left end of the region.

In [ ]:
# ---- Pauli matrices ----
const PAULI = Dict('I' => ComplexF64[1 0; 0 1], 'X' => ComplexF64[0 1; 1 0],
                   'Y' => ComplexF64[0 -im; im 0], 'Z' => ComplexF64[1 0; 0 -1])
# (Row/column 1 of each matrix is spin up, Z = +1; row/column 2 is spin down, Z = −1.)

"""
A Pauli string on k qubits, e.g. pauli(3, [(1, 'X'), (2, 'X')]) = X ⊗ X ⊗ I.
Qubits not listed get the identity.  kron(A, B, C): A acts on qubit 1, the most significant bit.
"""
function pauli(k, ops)
  factors = [PAULI['I'] for _ in 1:k]
  for (q, P) in ops
    factors[q] = PAULI[P]
  end
  return reduce(kron, factors)
end

"""
The neuron's terms on a region whose qubits are the chain sites `sites` (e.g. [24, 25, 26]):
the identity (bias), Z on every qubit, then ZZ, XX and YY on every neighbouring pair.
Names use chain-site numbers, e.g. "Z25" or "X24X25".  Returns (names, matrices).
"""
function neuron_terms(sites)
  k = length(sites)
  names, ops = ["I"], [Tuple{Int,Char}[]]
  for q in 1:k
    push!(names, "Z$(sites[q])"); push!(ops, [(q, 'Z')])
  end
  for P in "ZXY", q in 1:(k - 1)
    push!(names, "$P$(sites[q])$P$(sites[q+1])"); push!(ops, [(q, P), (q + 1, P)])
  end
  return names, [pauli(k, o) for o in ops]
end

"H(θ) = Σ_j θ_j H_j."
hamiltonian(terms, theta) = Hermitian(sum(t .* Hj for (t, Hj) in zip(theta, terms)))

### Output, loss and exact gradient
In the eigenbasis of H(θ) the output is simple: apply GReLU to each energy E_a, and weight it by how much of ρ sits on that eigenvector, ⟨a|ρ|a⟩. The gradient comes from the Daleckii–Krein formula for differentiating a function of a matrix.

In [ ]:
# ---- the neuron's output on each state ----
"Neuron outputs Tr[GReLU_T(H) ρ] for every state in `rhos`."
function outputs(terms, theta, rhos, T)
  E, V = eigen(hamiltonian(terms, theta))       # energies E and eigenvectors V of H(θ)
  g = grelu.(E, T)                              # activation of each energy
  # diag(V' ρ V) = how much of ρ sits on each eigenvector; the output is the weighted sum of activations
  return [real(dot(g, diag(V' * ρ * V))) for ρ in rhos]
end

# ---- exact loss and gradient for Algorithm 8 ----
"""
Divided differences of GReLU between every pair of energies:
G[a, b] = (f(E_a) − f(E_b)) / (E_a − E_b), and the derivative f'(E_a) = Φ(E_a/T) when E_a ≈ E_b.
"""
function divided_differences(E, T)
  G = zeros(length(E), length(E))
  for b in eachindex(E), a in eachindex(E)
    G[a, b] = abs(E[a] - E[b]) < 1e-9 ? Phi((E[a] + E[b]) / 2T) :
                                        (grelu(E[a], T) - grelu(E[b], T)) / (E[a] - E[b])
  end
  return G
end

"""
Squared loss and its exact gradient with respect to every θ_j.
  loss       = mean over samples of (output − target)²
  gradient_j = Tr[ H_j  V (G ∘ V† R V) V† ],   R = (2/M) Σ_m (output_m − target_m) ρ_m
"""
function loss_grad(terms, theta, rhos, targets, T)
  E, V = eigen(hamiltonian(terms, theta))
  out = [real(dot(grelu.(E, T), diag(V' * ρ * V))) for ρ in rhos]
  r = out .- targets                                       # how far each output is from its target
  R = sum((2 * r[m] / length(rhos)) .* rhos[m] for m in eachindex(rhos))   # residual-weighted mix of states
  K = V * (divided_differences(E, T) .* (V' * R * V)) * V'
  return mean(abs2, r), [real(dot(Hj, K)) for Hj in terms]
end

### Firing (Algorithm 5)
On hardware the output is not read off exactly. Each **firing** uses one copy of ρ: it picks an eigenvalue E of H with probability ⟨E|ρ|E⟩ and returns ReLU(E + T z), with z a standard normal number. The average of many firings is the neuron output.

In [ ]:
# ---- firing the neuron (Algorithm 5): one copy of ρ per firing ----
"One firing: pick energy E_a with probability ⟨a|ρ|a⟩ (via its cumulative sum `cdf`), return ReLU(E_a + T z)."
function fire(E, cdf, T, rng)
  a = min(searchsortedfirst(cdf, rand(rng)), length(E))
  return max(E[a] + T * randn(rng), 0.0)
end

"Classify one state from `ncopies` firings: s = +1 if the average firing is above 1/2."
function classify_by_firing(E, pop, T, ncopies, rng)
  cdf = cumsum(max.(pop, 0) ./ sum(max.(pop, 0)))
  return mean(fire(E, cdf, T, rng) for _ in 1:ncopies) > 0.5 ? 1 : -1
end

## 2. The data, and getting the index order right

The region states were cut from the full-chain files by the dataset's own tool, `make_regions.py` (run automatically in Section 3 if the file is missing). It writes ρ as a stack `rho[sample, row, column]`, with row = ket. Two conversions are needed before Julia can use it, and getting either wrong fails **silently**:

1. **Axis order.** Julia's HDF5 reads every array with its axes reversed, so each matrix arrives **transposed**. For a density matrix, transposing is the same as complex conjugation, and that swaps the labels s = +1 and s = −1 at t = 0.
2. **Bit order.** In the file, region qubit q (q = 0, 1, 2 for sites 24, 25, 26) is bit q of the matrix index, counting from the **least** significant bit. Our `kron`-built Pauli strings put the first qubit in the **most** significant bit. Converting means reversing the bits of every row and column index.

After converting, the loader is checked against numbers the simulation stored in each full-chain file. These include ⟨Z⟩ on every chain site, which differs between sites 24 and 26, so a reversed bit order would fail.

In [ ]:
# data.jl — load the region states, put their indices in the right order, check them, and split train/test.

using HDF5, LinearAlgebra, Printf, Random

In [ ]:
# ---- conversion 1: axis order ----
# Python wrote rho[sample, ket, bra]. Julia reads the axes backwards, as r[bra, ket, sample].
# Swapping the first two axes gives ρ[ket, bra, sample]: row = ket, the usual way to write a density matrix.
read_rho_stack(dset) = permutedims(read(dset), (2, 1, 3))

# ---- conversion 2: bit order ----
# File order ("little-endian"):  index = b_0 + 2 b_1 + 4 b_2      (b_q = 0 for spin up on region qubit q)
# kron order ("big-endian"):     index = 4 b_0 + 2 b_1 + b_2
# Going from one to the other reverses the k bits of the index (0-based).
reverse_bits(i, k) = sum(((i >> q) & 1) << (k - 1 - q) for q in 0:(k - 1))

"Re-index a 2^k × 2^k matrix from file (little-endian) order to kron (big-endian) order."
function to_kron_order(ρ, k)
  p = [reverse_bits(i, k) for i in 0:(2^k - 1)] .+ 1   # new index -> old index (+1: Julia counts from 1)
  return ρ[p, p]
end

### Loading
`Regions` holds one region's states. All of them use the same three qubits (one value of d).

In [ ]:
struct Regions
  sites::Vector{Int}                  # chain sites of the region, e.g. [24, 25, 26]
  rho::Vector{Matrix{ComplexF64}}     # the region states, in kron order
  s::Vector{Int}                      # the label: +1 or −1 (direction of the turn on qubit 25)
  t::Vector{Float64}                  # evolution time
  cell::Vector{Int}                   # which time cell; the two labels of one time share it
  trace_distance::Vector{Float64}     # how different the two labels' states are at this time (0 to 1)
end

"Load a region file written by make_regions.py, keeping the times in [tmin, tmax]."
function load_regions(path; tmin=-Inf, tmax=Inf)
  h5open(path, "r") do f
    k = Int(read(attributes(f)["k"]))
    first_site = read(f["first_site"])
    all(==(first_site[1]), first_site) || error("this loader expects one region (a single value of d)")
    t = read(f["t"])
    keep = findall(x -> tmin - 1e-6 <= x <= tmax + 1e-6, t)    # times are compared with a small tolerance
    rho = read_rho_stack(f["rho"])                             # conversion 1 (axis order)
    return Regions(collect(Int(first_site[1]) .+ (0:(k - 1))),
                   [to_kron_order(rho[:, :, m], k) for m in keep],   # conversion 2 (bit order)
                   Int.(read(f["s"])[keep]), t[keep], Int.(read(f["pair_id"])[keep]),
                   read(f["trace_distance"])[keep])
  end
end

### Checks
Each check prints its largest deviation and stops the pipeline if it is above 10⁻⁸.
- **(a)** Every ρ is a proper density matrix.
- **(b)** ⟨Z⟩ on each region qubit equals the ⟨Z⟩ the simulation stored for that chain site. This checks the **bit order**.
- **(c)** For the three qubits 24–26, ρ equals the copy the simulation stored itself. This checks the **axis order** (no transpose or conjugation).
- **(d)** The two labels are spin flips of each other, as the physics requires: ρ₋ = X⊗X⊗X ρ₊ X⊗X⊗X.

In [ ]:
"Path of the full-chain state file for one (Hamiltonian, label, time), e.g. XXX/XXX_s+1_t06.400.h5."
mps_file(dir, ham, s, t) = joinpath(dir, ham, @sprintf("%s_s%+d_t%06.3f.h5", ham, s, t))

"Run checks (a)-(d) on the loaded states.  `mps_dir` is the dataset folder holding XX/ and XXX/."
function check_regions(R::Regions, mps_dir, ham)
  k = length(R.sites)
  report(name, dev) = (@printf("  %-58s largest deviation %.1e\n", name, dev);
                       dev < 1e-8 || error("check failed: $name"))

  # (a) proper density matrices: Hermitian, trace 1, no negative eigenvalues
  report("(a) ρ is Hermitian", maximum(norm(ρ - ρ') for ρ in R.rho))
  report("(a) trace of ρ is 1", maximum(abs(tr(ρ) - 1) for ρ in R.rho))
  report("(a) no negative eigenvalues", max(0.0, -minimum(minimum(eigvals(Hermitian(ρ))) for ρ in R.rho)))

  # (b) bit order: ⟨Z_q⟩ from our ρ and our kron-order Z  vs  ⟨Z⟩ the simulation stored for that chain site
  Zs = [pauli(k, [(q, 'Z')]) for q in 1:k]
  dev_b, gap = 0.0, 0.0
  for m in eachindex(R.rho)
    sz = h5open(f -> read(f["checks/sz"]), mps_file(mps_dir, ham, R.s[m], R.t[m]))   # ⟨Z⟩ on sites 1..50
    ours = [real(tr(Z * R.rho[m])) for Z in Zs]
    dev_b = max(dev_b, maximum(abs.(ours .- sz[R.sites])))
    gap = max(gap, abs(sz[R.sites[1]] - sz[R.sites[end]]))
  end
  report("(b) ⟨Z⟩ per qubit matches the stored chain values", dev_b)
  @printf("      (for scale: ⟨Z⟩ on site %d and site %d differ by up to %.2f, so a reversed order would fail)\n",
          R.sites[1], R.sites[end], gap)

  # (c) axis order: compare with the 3-qubit ρ on sites 24-26 that the simulation stored (checks/rho_k3_d0).
  #     That copy is also little-endian and also arrives transposed in Julia, so it gets the same two conversions.
  if R.sites == [24, 25, 26]
    dev_c = 0.0
    for m in eachindex(R.rho)
      stored = h5open(f -> read(f["checks/rho_k3_d0"]), mps_file(mps_dir, ham, R.s[m], R.t[m]))
      dev_c = max(dev_c, maximum(abs.(to_kron_order(transpose(stored), 3) .- R.rho[m])))
    end
    report("(c) ρ matches the simulation's own copy (sites 24-26)", dev_c)
  end

  # (d) spin flip: within each time cell, ρ(s = −1) = X⊗X⊗X ρ(s = +1) X⊗X⊗X
  F = pauli(k, [(q, 'X') for q in 1:k])
  dev_d = 0.0
  for c in unique(R.cell)
    p = findfirst(m -> R.cell[m] == c && R.s[m] == 1, eachindex(R.s))
    n = findfirst(m -> R.cell[m] == c && R.s[m] == -1, eachindex(R.s))
    dev_d = max(dev_d, maximum(abs.(F * R.rho[p] * F .- R.rho[n])))
  end
  report("(d) the two labels are each other's spin flip", dev_d)
end

### Train / test split by time
Both labels at one time are exact spin flips of each other. Putting one in training and the other in test would hand the neuron the answer, so whole times are held out: 20% of the times, chosen at random, go to the test set with both their labels.

In [ ]:
"Hold out whole time cells (both labels together). Returns the indices of the training and test samples."
function split_by_cell(R::Regions; test_fraction=0.2, seed=20261007)
  cells = sort(unique(R.cell))
  test_cells = Set(shuffle(MersenneTwister(seed), cells)[1:round(Int, test_fraction * length(cells))])
  is_test = [c in test_cells for c in R.cell]
  return findall(.!is_test), findall(is_test)
end

## 3. The pipeline

### Settings

In [ ]:
using Optimisers, Printf

# ---- which data ----
const DATA_DIR = joinpath(HERE, "..", "data", "source_sign_mps_N50")   # the full-chain dataset
const HAM = "XXX"              # Hamiltonian whose states we classify
const K, D = 3, 0              # region: K qubits centred D sites from the turned qubit 25 -> sites 24, 25, 26
const TMIN, TMAX = 0.1, 7.5    # times used. t = 0 is left out (see below), and XXX's signal reaches the chain
                               # ends at t ≈ 7.6, after which reflections mix in.
const REGION_FILE = joinpath(HERE, "regions", "$(HAM)_k$(K)_d$(D).h5")
const RESULTS = mkpath(joinpath(HERE, "results"))

# ---- training settings (the same as the XX vs XXX minimal pipeline) ----
const T = 1.0                                  # width of the GReLU smoothing
const L2 = 1e-4                                # weight decay
const STEPS, LR, SEED = 300, 0.05, 20261007    # Adam steps, learning rate, random seed

"Write rows of values to a CSV file with the given header."
writecsv(path, header, rows) = open(io -> (println(io, join(header, ","));
                                           foreach(r -> println(io, join(r, ",")), rows)), path, "w")

### Cut the region states (first run only)
The dataset's own tool cuts the three-qubit states out of the 262 full-chain XXX files: every time and both labels. It takes about 30 s and writes `regions/XXX_k3_d0.h5`. Nothing in the data folder is changed.

In [ ]:
if !isfile(REGION_FILE)
  mkpath(dirname(REGION_FILE))
  run(Cmd(`python3 make_regions.py --hamiltonian $HAM --k $K --d $D --workers 4 --out $REGION_FILE`; dir=DATA_DIR))
end

### Load and check

In [ ]:
R = load_regions(REGION_FILE; tmin=TMIN, tmax=TMAX)
@printf("%d states: %s, sites %s, t = %.1f to %.1f, %d times × 2 labels\n",
        length(R.s), HAM, R.sites, minimum(R.t), maximum(R.t), length(unique(R.cell)))
check_regions(R, DATA_DIR, HAM)

### Why t = 0 is left out
At t = 0 the two labels' states are complex conjugates of each other, ρ₋ = ρ₊*. All of the neuron's terms are real matrices, so H(θ) and GReLU(H(θ)) are real too. For any real matrix A, Tr[A ρ*] = Tr[A ρ]. The two labels therefore give **exactly the same output**, for every θ, even though the states themselves are very different. The cell below shows this with random weights. Telling them apart needs an imaginary term such as the spin current X₂₄Y₂₅ − Y₂₄X₂₅.

In [ ]:
R0 = load_regions(REGION_FILE; tmin=0.0, tmax=0.0)
ρp, ρm = R0.rho[findfirst(==(1), R0.s)], R0.rho[findfirst(==(-1), R0.s)]
term_names, terms = neuron_terms(R.sites)
θ_random = randn(MersenneTwister(1), length(terms))
o = outputs(terms, θ_random, [ρp, ρm], T)
@printf("t = 0:  ρ₋ = conj(ρ₊) to %.1e;  trace distance between them %.2f;  outputs with random θ: %.6f vs %.6f\n",
        maximum(abs.(ρm .- conj(ρp))), R0.trace_distance[1], o...)

### Split and targets

In [ ]:
train, test = split_by_cell(R; seed=SEED)
@printf("train %d states (%d times),  test %d states (%d times)\n",
        length(train), length(unique(R.cell[train])), length(test), length(unique(R.cell[test])))

target(s) = s == 1 ? 1.0 : 0.0                          # Algorithm 8 targets: 1 for s = +1, 0 for s = −1
accuracy(out, idx) = mean((out .> 0.5) .== (R.s[idx] .== 1))

### Train the neuron and record θ
Adam runs on loss + (λ/2)|θ|² with the exact gradient. **θ and the gradient are stored at every step.** Loss and accuracies are printed every 20 steps.

In [ ]:
"""
Adam on the squared loss (+ weight decay) with the exact gradient.
Returns the final weights, θ and the loss gradient at every step (column s+1 = step s), and a history.
"""
function train_neuron(terms)
  rhos, targets = R.rho[train], target.(R.s[train])
  theta = 0.1 .* randn(MersenneTwister(SEED), length(terms))      # small random starting weights
  opt = Optimisers.setup(Optimisers.Adam(LR), theta)
  thetas, grads = zeros(length(terms), STEPS + 1), zeros(length(terms), STEPS + 1)
  history = []
  for step in 0:STEPS
    L, g = loss_grad(terms, theta, rhos, targets, T)               # loss and exact gradient at the current θ
    thetas[:, step + 1], grads[:, step + 1] = theta, g             # record where θ is and where it is pushed
    if step % 20 == 0
      acc_train = accuracy(outputs(terms, theta, R.rho[train], T), train)
      acc_test = accuracy(outputs(terms, theta, R.rho[test], T), test)
      push!(history, (step, L, acc_train, acc_test))
      @printf("  step %3d   train loss %.4f   train accuracy %.3f   test accuracy %.3f   ||θ||_1 %.2f\n",
              step, L, acc_train, acc_test, norm(theta, 1))
    end
    step == STEPS && break
    opt, theta = Optimisers.update!(opt, theta, g .+ L2 .* theta)  # one Adam step
  end
  return (; theta=copy(theta), thetas, grads, history)
end

model = train_neuron(terms)

writecsv(joinpath(RESULTS, "history.csv"), ["step", "train_loss", "train_accuracy", "test_accuracy"], model.history)
writecsv(joinpath(RESULTS, "weights.csv"), ["term", "theta"], zip(term_names, model.theta))
writecsv(joinpath(RESULTS, "theta_trajectory.csv"), ["step", "term", "theta", "grad"],
         [(st, term_names[j], model.thetas[j, st + 1], model.grads[j, st + 1]) for st in 0:STEPS for j in eachindex(term_names)])

## 4. Testing

### Exact outputs
The output of every state, train and test, is saved together with its time and label.

In [ ]:
out = outputs(terms, model.theta, R.rho, T)
@printf("exact outputs:  train accuracy %.3f   test accuracy %.3f   ||θ||_1 = %.2f\n",
        accuracy(out[train], train), accuracy(out[test], test), norm(model.theta, 1))

set_of(m) = m in test ? "test" : "train"
writecsv(joinpath(RESULTS, "outputs.csv"), ["t", "s", "set", "output", "trace_distance"],
         [(R.t[m], R.s[m], set_of(m), out[m], R.trace_distance[m]) for m in eachindex(R.s)])

### Classify by firing
Each test state is classified from N firings (N copies of the state), and the accuracy is averaged over 20 repeats.

In [ ]:
const COPIES = [1, 4, 16, 64, 256, 1024, 4096]
const REPS = 20
rng = MersenneTwister(SEED + 1)
E, V = eigen(hamiltonian(terms, model.theta))
pops = [real(diag(V' * R.rho[m] * V)) for m in test]      # each test state's weight on each eigenvector of H
firing = map(COPIES) do N
  mean(mean(classify_by_firing(E, pops[j], T, N, rng) == R.s[m] for (j, m) in enumerate(test)) for _ in 1:REPS)
end
println(join((@sprintf("%d copies: %.3f", N, a) for (N, a) in zip(COPIES, firing)), "   "))
writecsv(joinpath(RESULTS, "firing.csv"), ["copies", "accuracy"], zip(COPIES, firing))

### The data over the whole time range (for the overview figure)
⟨Z⟩ on each of the three qubits and the trace distance at every stored time, including t = 0 and the times after the reflections.

In [ ]:
Rall = load_regions(REGION_FILE)
Zq = [pauli(K, [(q, 'Z')]) for q in 1:K]
writecsv(joinpath(RESULTS, "overview.csv"), ["t", "s", "trace_distance", ["z$(x)" for x in Rall.sites]...],
         [(Rall.t[m], Rall.s[m], Rall.trace_distance[m], [real(tr(Z * Rall.rho[m])) for Z in Zq]...)
          for m in eachindex(Rall.s)])

## 5. Figures

The figures are drawn from the CSVs in `results/`, so this section can be re-run on its own after the first cell. Each figure is shown inline and saved to `figures/`.

In [ ]:
using Plots, Statistics

const RES = joinpath(HERE, "results")
const FIG = mkpath(joinpath(HERE, "figures"))

"Read a CSV written above into named columns (numbers parsed where possible)."
function readcsv(name)
  lines = readlines(joinpath(RES, name))
  head = Symbol.(split(lines[1], ","))
  rows = [split(l, ",") for l in lines[2:end]]
  col(i) = (v = getindex.(rows, i); all(x -> tryparse(Float64, x) !== nothing, v) ? parse.(Float64, v) : String.(v))
  return NamedTuple{Tuple(head)}(Tuple(col(i) for i in eachindex(head)))
end

termkind(n) = n == "I" ? "I" : count(isletter, n) == 1 ? "Z" : string(n[1], n[1])   # "X24X25" -> "XX"
const KINDS = ["I", "Z", "ZZ", "XX", "YY"]
const KCOLOR = Dict("I" => :black, "Z" => :seagreen, "ZZ" => :crimson, "XX" => :royalblue, "YY" => :darkorange)
const SCOLOR = Dict(1.0 => :crimson, -1.0 => :royalblue)      # label colours: s = +1 red, s = −1 blue
default(framestyle=:box, grid=false, dpi=200)

### The data
Left: the trace distance between the two labels' states (0 = identical, 1 = perfectly distinguishable). The shaded band is the window used for training. Right: ⟨Z⟩ on each qubit for s = +1; for s = −1 every curve is exactly mirrored.

In [ ]:
ov = readcsv("overview.csv")
p = ov.s .== 1
p1 = plot(ov.t[p], ov.trace_distance[p], lw=2, c=:black, label="trace distance", ylim=(0, 1.02),
          xlabel="time t", ylabel="", title="distinguishability of the two labels", legend=:topright)
vspan!(p1, [0.1, 7.5], c=:gold, alpha=0.15, label="training window")
zcols = filter(n -> startswith(String(n), "z"), collect(keys(ov)))
p2 = plot(xlabel="time t", ylabel="⟨Z⟩ for s = +1", title="⟨Z⟩ on each qubit", legend=:topright)
for (i, zc) in enumerate(zcols)
  plot!(p2, ov.t[p], getproperty(ov, zc)[p], lw=2, c=get(cgrad(:viridis), (i - 1) / max(length(zcols) - 1, 1) * 0.85),
        label="site " * String(zc)[2:end])
end
hline!(p2, [0], c=:gray70, lw=0.5, label="")
vspan!(p2, [0.1, 7.5], c=:gold, alpha=0.15, label="")
fig = plot(p1, p2, layout=(1, 2), size=(1000, 360), margin=5Plots.mm)
savefig(fig, joinpath(FIG, "0_data.png"))
fig

### Training loss and accuracy

In [ ]:
h = readcsv("history.csv")
p1 = plot(h.step, h.train_loss, lw=2, c=:darkorange, yscale=:log10, xlabel="Adam step",
          ylabel="training loss", title="squared loss", legend=false)
p2 = plot(h.step, h.train_accuracy, lw=2, c=:darkorange, label="train", ylim=(0.4, 1.03),
          xlabel="Adam step", ylabel="accuracy (exact outputs)", title="accuracy", legend=:bottomright)
plot!(p2, h.step, h.test_accuracy, lw=2, c=:darkorange, ls=:dash, label="test")
fig = plot(p1, p2, layout=(1, 2), size=(900, 350), margin=5Plots.mm)
savefig(fig, joinpath(FIG, "1_training.png"))
fig

### How θ moves during training
One panel per kind of term, and ‖θ‖₁.

In [ ]:
tr_ = readcsv("theta_trajectory.csv")
steps = sort(unique(tr_.step))
termnames = tr_.term[tr_.step .== 0]
traj(n) = tr_.theta[tr_.term .== n]               # rows are written in step order
ps = map(KINDS) do k
  ns = filter(n -> termkind(n) == k, termnames)
  q = plot(xlabel="Adam step", ylabel="θ", title=k == "I" ? "bias (identity)" : "$k terms", legend=:best,
           legendfontsize=7)
  hline!(q, [0], c=:gray70, lw=0.5, label="")
  for (i, n) in enumerate(ns)
    plot!(q, steps, traj(n), lw=1.8, label=n,
          c=k == "I" ? :black : get(cgrad(:viridis), (i - 1) / max(length(ns) - 1, 1) * 0.85))
  end
  q
end
l1 = [sum(abs, tr_.theta[tr_.step .== st]) for st in steps]
push!(ps, plot(steps, l1, lw=2, c=:black, xlabel="Adam step", ylabel="‖θ‖₁", title="‖θ‖₁", legend=false))
fig = plot(ps..., layout=(2, 3), size=(1050, 600), margin=4Plots.mm)
savefig(fig, joinpath(FIG, "2_theta_trajectories.png"))
fig

### Final weights

In [ ]:
w = readcsv("weights.csv")
lim = 1.15 * maximum(abs, w.theta)
p = plot(ylabel="θ", title="trained weights", size=(700, 380), legend=:outerright, ylim=(-lim, lim),
         xticks=(1:length(w.term), w.term), xrotation=45, bottom_margin=6Plots.mm, left_margin=4Plots.mm)
for k in KINDS
  sel = termkind.(w.term) .== k
  bar!(p, findall(sel), w.theta[sel], bar_width=0.8, c=KCOLOR[k], lc=KCOLOR[k], label=k)
end
hline!(p, [0], c=:black, lw=0.5, label="")
savefig(p, joinpath(FIG, "3_weights.png"))
p

### The neuron's output on every state
Output − ½ against time; above the dashed line the neuron says s = +1. Filled dots are training states and open circles are test states.

In [ ]:
o = readcsv("outputs.csv")
p = plot(xlabel="time t", ylabel="output − ½", title="neuron output (red s = +1, blue s = −1)",
         size=(750, 380), legend=:topright)
for (sv, lab) in ((1.0, "s = +1"), (-1.0, "s = −1"))
  r1 = (o.s .== sv) .& (o.set .== "train")
  r2 = (o.s .== sv) .& (o.set .== "test")
  scatter!(p, o.t[r1], o.output[r1] .- 0.5, ms=3, msw=0, c=SCOLOR[sv], label=lab * " train")
  scatter!(p, o.t[r2], o.output[r2] .- 0.5, ms=5, mc=:white, msc=SCOLOR[sv], msw=1.5, label=lab * " test")
end
hline!(p, [0], ls=:dash, c=:black, label="")
savefig(p, joinpath(FIG, "4_outputs.png"))
p

### Classifying by firing: accuracy against copies

In [ ]:
f = readcsv("firing.csv")
p = plot(f.copies, f.accuracy, lw=2, marker=:circle, c=:darkorange, xscale=:log2, label="fired neuron",
         xlabel="copies of each test state", ylabel="test accuracy", ylim=(0.45, 1.03),
         title="classifying by firing the neuron (Algorithm 5)", size=(600, 380), legend=:bottomright)
hline!(p, [0.5], ls=:dash, c=:gray, label="chance")
savefig(p, joinpath(FIG, "5_firing.png"))
p